# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alikadirguzel/flyrankinternship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I am on **Lane 2 — Refresh / Content Opportunity Scoring**.

**Task type: ranking / scoring.** The question this lane answers is "which pages first?", not "what kinds of pages exist?" (clustering) and not a yes/no for every URL as the product itself. An editor with a 50-page sprint needs an ordered queue, not 30,000 independent labels.

I may still train a classifier underneath — a probability that a page looks like a decline/opportunity case — but that probability only becomes useful when it is turned into a **priority score** and sorted. The decision the output supports is weekly editorial allocation: review, refresh, expand, protect, or keep monitoring.

Mapped onto the ML loop from the live session:

| Loop step | In this lane |
|---|---|
| World / data | 30,000-page starter snapshot of trailing-90-day search and engagement metrics (32 clients) |
| Features / representations | Observable signals known at review time: volume, position, CTR, freshness, depth, engagement. Never `trend_direction` or `trend_pct` |
| Model | A scorer that combines those signals |
| Output | Ranked review queue: score + suggested action + reason codes |
| Decision / action | Content strategist / SEO editor spends limited hours on the top-K pages |
| Changed world | Some pages get edited; later traffic may move. I will not claim the edit caused the move |

**One-paragraph frame.** For content strategists and SEO editors, deciding which published pages to review first in a limited refresh sprint, we will build a ranked priority queue from the starter content snapshot (and later warehouse daily facts), scoring refresh-review priority against a decline/opportunity proxy, measured by Precision@50. A wrong call costs wasted editor hours on a healthy page, or a missed high-visibility decay. A plain rule is not enough because visibility, freshness, CTR-at-position, and engagement interact, and a single cutoff either floods the queue or misses almost everyone. We will claim only observed, directional, decision-support results.


In [1]:
from pathlib import Path
import pandas as pd

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("flyrankinternship/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    if not Path("flyrankinternship").exists():
        get_ipython().system("git clone https://github.com/alikadirguzel/flyrankinternship.git")
    data_path = Path("flyrankinternship/data/raw/content_refresh_anonymized.csv")

df_raw = pd.read_csv(data_path)

print(f"loaded: {data_path}")
print(f"rows x cols: {df_raw.shape[0]:,} x {df_raw.shape[1]}")
print(f"unique content_id (pages): {df_raw['content_id'].nunique():,}")
print(f"unique client_id: {df_raw['client_id'].nunique():,}")
print(f"duplicate content_id rows: {int(df_raw['content_id'].duplicated().sum())}")
print()
print("task type this lane maps to: ranking / scoring")
print("decision the output supports: which page an editor reviews first")
print("editor capacity assumed for the metric: top 50")
print()
print("trend_direction mix (this is a current-window bucket, not a feature):")
print(df_raw["trend_direction"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")



loaded: data\raw\content_refresh_anonymized.csv
rows x cols: 30,000 x 44
unique content_id (pages): 30,000
unique client_id: 32
duplicate content_id rows: 0

task type this lane maps to: ranking / scoring
decision the output supports: which page an editor reviews first
editor capacity assumed for the metric: top 50

trend_direction mix (this is a current-window bucket, not a feature):
trend_direction
down      54.21%
stable    19.87%
up        14.63%
new        7.45%
flat       3.84%
Name: proportion, dtype: object


## 2. Target or proxy

**What the editor sees:** a continuous **refresh-priority score**. Higher means "look at this page sooner."

**What I can sketch on the starter CSV today** is a **proxy label**, not a future outcome:

```text
is_declining_label = (trend_direction == "down")
```

That 0/1 column is **defined by a rule on the current window**. `trend_direction` is computed from `trend_pct`, which compares impressions in the last 30 days vs the previous 30 days (`down` means a drop worse than −20%). So the label is observed *as a bucket of current movement*, but it is not an independent later-window outcome. If `trend_direction` or `trend_pct` ever enter the feature list, the model just relearns the rule.

**What I actually want later (warehouse daily facts):** features from a *prior* window → an outcome measured in a *later* window, for example:

```text
signals from days 1–90  →  did impressions/clicks drop in days 91–120?
```

Until that contract exists, I will evaluate ranking quality against the starter proxy, and I will say so out loud. The code cell below adds `is_declining_label` next to the page id so the target column is a real series, not a sentence.


In [2]:
# Starter proxy: defined from the current-window trend bucket.
# Stronger later: a future-window outcome from warehouse daily facts.
df = df_raw.copy()
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

n = len(df)
n_pos = int(df["is_declining_label"].sum())
print("target column sketched: is_declining_label")
print("rule that defines it:  trend_direction == 'down'")
print("source of that bucket: trend_pct (last-30d vs prev-30d impressions)")
print("never use as features: trend_direction, trend_pct")
print()
print(f"positives (proxy=1): {n_pos:,} / {n:,}  ({n_pos / n:.1%})")
print(f"negatives (proxy=0): {n - n_pos:,} / {n:,}  ({1 - n_pos / n:.1%})")
print()
print("what one target value looks like next to the page it belongs to:")
target_sketch = df[
    ["content_id", "client_id", "impressions_90d", "trend_direction", "is_declining_label"]
].head(8)
display(target_sketch)

print("label vs trend_direction (the proxy is just the 'down' row):")
print(
    pd.crosstab(df["trend_direction"], df["is_declining_label"], margins=True)
)



target column sketched: is_declining_label
rule that defines it:  trend_direction == 'down'
source of that bucket: trend_pct (last-30d vs prev-30d impressions)
never use as features: trend_direction, trend_pct

positives (proxy=1): 16,262 / 30,000  (54.2%)
negatives (proxy=0): 13,738 / 30,000  (45.8%)

what one target value looks like next to the page it belongs to:


,content_id,client_id,impressions_90d,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,down,1
3,content_331d6c4de07b,client_19581e27de,11751,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,down,1
6,content_9a34b442b552,client_8722616204,20,down,1
7,content_a63219c6e95a,client_19581e27de,1724,stable,0


label vs trend_direction (the proxy is just the 'down' row):
is_declining_label      0      1    All
trend_direction                        
down                    0  16262  16262
flat                 1152      0   1152
new                  2236      0   2236
stable               5962      0   5962
up                   4388      0   4388
All                 13738  16262  30000


## 3. Success metric

**Primary metric: Precision@50.**

That is the share of the top 50 ranked pages that are positive on the chosen label. It matches the action: a reviewer can realistically open about 50 pages, not 30,000. Accuracy on the full inventory is the wrong question — with a ~54% decline base rate, a model that always says "declining" looks 54% accurate and is useless as a queue.

**What "good" means, before any training:**

- A random ranking would land near the **base rate** (~0.54 on this slice). That is the floor, not the goal.
- A **transparent rule baseline** (stale + visible, thin + visible, and similar if-statements) is the number I have to beat, on a **client-holdout** split so whole clients stay out of training.
- I will call the ranker useful only if Precision@50 is **clearly above that rule baseline** and the top of the list is inspectable (reason codes a human can disagree with).

**Secondary metrics** (supporting, not the decision metric): ROC-AUC and average precision for the score that feeds the ranker. Recall@50 matters if missing a decaying high-visibility page is costlier than wasting an hour — I will report it, but I will not optimize it alone.

I am **not** copying the reference pipeline's published Precision@50 (baseline 0.24 / random forest 0.74 on its holdout) as my result. Those numbers prove a learned ranker *can* beat a rule on this slice. Mine have to be recomputed.


In [3]:
import numpy as np

def precision_at_k(y_true, scores, k=50):
    ranked = pd.DataFrame({"y": y_true, "score": scores}).sort_values(
        "score", ascending=False, kind="mergesort"
    )
    top = ranked.head(k)
    return float(top["y"].mean()), int(top["y"].sum()), k

base_rate = float(df["is_declining_label"].mean())
print(f"base rate (random ranking's expected Precision@K): {base_rate:.3f}")
print("primary metric: Precision@50  — share of the top 50 that match the proxy")
print("good: beat a written rule baseline on a client-holdout split, not this in-sample peek")
print()

# Naive rankings I can compute today — none of these use trend_* as a feature.
rankings = {
    "random shuffle": pd.Series(np.random.default_rng(42).random(len(df)), index=df.index),
    "impressions_90d (high first)": df["impressions_90d"],
    "days_since_last_update (stale first)": df["days_since_last_update"],
}

rows = []
for name, scores in rankings.items():
    p_at, n_hit, k = precision_at_k(df["is_declining_label"], scores, k=50)
    rows.append(
        {
            "ranking": name,
            "Precision@50": round(p_at, 3),
            "hits_in_top_50": n_hit,
            "vs_base_rate": round(p_at - base_rate, 3),
        }
    )

metrics = pd.DataFrame(rows)
display(metrics)
print()
print("reading: ranking by volume or staleness on the full slice is not a trained model.")
print("it only proves Precision@50 is computable today, and that a naive sort is not enough.")



base rate (random ranking's expected Precision@K): 0.542
primary metric: Precision@50  — share of the top 50 that match the proxy
good: beat a written rule baseline on a client-holdout split, not this in-sample peek



,ranking,Precision@50,hits_in_top_50,vs_base_rate
0,random shuffle,0.44,22,-0.102
1,impressions_90d (high first),0.42,21,-0.122
2,days_since_last_update (stale first),0.52,26,-0.022



reading: ranking by volume or staleness on the full slice is not a trained model.
it only proves Precision@50 is computable today, and that a naive sort is not enough.


## 4. The unit of analysis, as a real dataframe

**One row = one published content item (page)** for one client.

Grain: `content_id` (page) nested under `client_id` (site/account). The starter file is already at that grain — 30,000 unique pages, 32 clients, no duplicate `content_id`. IDs are pseudonyms: they are for grouping and splits, never features.

**Lane slice (Refresh / Content Opportunity Scoring):** pages that are old enough to have a 90-day history and that actually appeared in search. The starter pipeline's filters are `impressions_90d > 0` and `content_age_days >= 90`. In this CSV every row already passes both, so the "slice" is the full 30,000 rows — not a different table.

The dataframe below is that unit: each row is a page I could put on a review queue, with the proxy target sketched beside the signals an editor would actually look at. Rate columns are ×100 percentages (`ctr = 0.76` means 0.76%). `avg_position = 0` means missing position data, not rank zero.


In [4]:
# Lane slice: measurable search history + old enough for a 90-day window.
# In this starter CSV both filters already hold for every row.
lane = df[
    (df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)
].copy()

print("unit of analysis: one row = one content page (content_id) for one client")
print(f"lane slice rows: {len(lane):,}")
print(f"pages: {lane['content_id'].nunique():,}   clients: {lane['client_id'].nunique():,}")
print(f"duplicate pages in slice: {int(lane['content_id'].duplicated().sum())}")
print()
print("content_type mix:")
print(lane["content_type"].value_counts())
print()

unit_cols = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "word_count",
    "is_declining_label",
]
print("first 10 pages — this is the grain a ranked queue would sort:")
display(lane[unit_cols].head(10))

print()
print("ctr is a x100 percentage (0.76 means 0.76%, not 76%).")
print(f"avg_position == 0 (no position data, not rank zero): {(lane['avg_position'] == 0).sum():,}")



unit of analysis: one row = one content page (content_id) for one client
lane slice rows: 30,000
pages: 30,000   clients: 32
duplicate pages in slice: 0

content_type mix:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

first 10 pages — this is the grain a ranked queue would sort:


,content_id,client_id,content_type,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,10.6,0.76,20,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,20.3,0.05,25,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,36.5,0.09,20,3515.0,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,6.2,0.49,22,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,44.0,0.13,14,2803.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,8.5,0.03,20,3080.0,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,7.0,0.00,20,3059.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,21.2,0.06,22,NaN,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,46.0,0.09,20,3807.0,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,4.9,0.16,104,NaN,1



ctr is a x100 percentage (0.76 means 0.76%, not 76%).
avg_position == 0 (no position data, not rank zero): 1,205


## 5. Why ML beats a fixed rule here

A dashboard or a single if-statement is the right tool when one cutoff captures the decision. Here it does not.

The most obvious refresh rule — "high-visibility page not updated in 180 days" — flags **17 pages**. At the same time, **9,961** pages with ≥500 impressions are already tagged `down`. Sixteen of those 17 stale-visible pages are declining; the other 9,945 high-visibility declines fail the staleness cutoff. Age/freshness and current movement are not the same thing, so a freshness if-statement either misses almost the whole opportunity or, if you loosen it, floods the queue with noise.

The other signals are tangled in the same way:

- CTR only means something next to position (and position 0 is "no data").
- Thin word count on a visible page is a different action from a long page with weak engagement.
- Missing keyword fields follow `content_type` — a blind `fillna(0)` quietly encodes article type.
- Clients do not share one history depth; a global threshold is a policy, not a law.

That is why this is an ML / analysis problem: the pattern (pages losing search visibility while still having demand) is real, but it is spread across many shifting signals. A learned score can combine them; a ranker can be checked with Precision@50 against a written baseline. A rule still belongs in the system — as the **baseline** and as **reason codes** a human can audit — not as the final queue.

I will still not claim that ranking a page high *causes* recovery. The output is decision-support for which page to review first.


In [5]:
stale_visible = (lane["days_since_last_update"] >= 180) & (lane["impressions_90d"] >= 500)
high_vis_down = (lane["trend_direction"] == "down") & (lane["impressions_90d"] >= 500)

n_stale = int(stale_visible.sum())
n_down = int(high_vis_down.sum())
n_both = int((stale_visible & high_vis_down).sum())
n_stale_only = int((stale_visible & ~high_vis_down).sum())
n_down_not_stale = int((high_vis_down & ~stale_visible).sum())

print("fixed rule: days_since_last_update >= 180 AND impressions_90d >= 500")
print(f"pages the rule flags: {n_stale:,}")
print(f"high-visibility pages already tagged down: {n_down:,}")
print()
print(f"rule AND declining:     {n_both:,}")
print(f"rule but NOT declining: {n_stale_only:,}")
print(f"declining, rule misses: {n_down_not_stale:,}")
print()
print("a 180-day freshness cutoff cannot be the queue — it misses almost every visible decline.")
print()

# CTR vs position: same CTR means different things on page 1 vs page 3.
pos_ok = lane[lane["avg_position"] > 0].copy()
pos_ok["position_bucket"] = pd.cut(
    pos_ok["avg_position"],
    bins=[0, 3, 10, 20, 50, 1_000],
    labels=["top_3", "page_1", "11-20", "21-50", "deep"],
)
ctr_by_pos = (
    pos_ok.groupby("position_bucket", observed=True)["ctr"]
    .median()
    .round(2)
    .rename("median_ctr_x100")
)
print("median CTR (x100) by position bucket — why a global ctr < 0.5 rule is blunt:")
display(ctr_by_pos.to_frame())



fixed rule: days_since_last_update >= 180 AND impressions_90d >= 500
pages the rule flags: 17
high-visibility pages already tagged down: 9,961

rule AND declining:     16
rule but NOT declining: 1
declining, rule misses: 9,945

a 180-day freshness cutoff cannot be the queue — it misses almost every visible decline.

median CTR (x100) by position bucket — why a global ctr < 0.5 rule is blunt:


,median_ctr_x100
position_bucket,
top_3,0.00
page_1,0.16
11-20,0.10
21-50,0.03
deep,0.00


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Frame recap (for the person who asks "what decision does this improve?"):** it improves *which page an editor reviews first*, using a ranked score, evaluated by Precision@50 against a proxy decline label, without claiming that a refresh causes recovery.
